In [ ]:
# %%capture
# !pip install pandas numpy seaborn plotly

# ДЗ 1 — Постановка задачи, выбор метрики, EDA

**Студентка:** Григорьева Арина Анатольевна

**Курс:** Машинное обучение и нейросети, ИТМО, весна 2026

---
## Задание 1. Постановка задачи

### 1.1 Бизнес-постановка задачи

**Задача:** по результатам клинического обследования пациента (возраст, пол, тип боли в груди, уровень холестерина, максимальная ЧСС и др.) предсказать наличие или отсутствие ишемической болезни сердца.

**Практическое применение:** подобная модель может применяться в кардиологических клиниках и системах телемедицины как инструмент поддержки принятия решений — позволяет быстро выделить пациентов из группы высокого риска и направить их на дополнительные обследования (ЭКГ, ЭхоКГ, коронарографию) ещё до появления выраженных симптомов.

### 1.2 ML-постановка задачи

**Тип задачи:** бинарная классификация.

- **Целевая переменная:** `target` — бинарная переменная: `1` (болезнь сердца есть) или `0` (болезни сердца нет)
- **Признаки:** 13 клинических показателей пациента (возраст, пол, тип боли в груди, давление, холестерин, ЧСС и др.)
- **Объекты:** пациенты кардиологической клиники Кливленда, прошедшие коронарографию

### 1.3 Набор данных

Используется датасет **Heart Disease Cleveland** (UCI Machine Learning Repository).

| Признак | Описание |
|---|---|
| `age` | Возраст пациента (лет) |
| `sex` | Пол: 1 = мужчина, 0 = женщина |
| `cp` | Тип боли в груди: 0=типичная стенокардия, 1=атипичная, 2=неангинальная, 3=бессимптомная |
| `trestbps` | Артериальное давление в покое (мм рт. ст.) |
| `chol` | Уровень холестерина в сыворотке крови (мг/дл) |
| `fbs` | Сахар натощак > 120 мг/дл: 1 = да, 0 = нет |
| `restecg` | Результат ЭКГ в покое: 0 = норма, 1 = аномалия ST-T, 2 = гипертрофия ЛЖ |
| `thalach` | Максимальная ЧСС при нагрузочном тесте (уд./мин) |
| `exang` | Стенокардия при нагрузке: 1 = да, 0 = нет |
| `oldpeak` | Депрессия ST при нагрузке относительно покоя |
| `slope` | Наклон пикового сегмента ST: 0 = подъём, 1 = плоский, 2 = снижение |
| `ca` | Количество крупных сосудов (0–3), окрашенных при флуороскопии |
| `thal` | Таллиевый стресс-тест: 1 = норма, 2 = фиксированный дефект, 3 = обратимый дефект |
| `target` | Целевая переменная: 1 = болезнь сердца, 0 = норма |

Датасет содержит **303 записи** и **13 признаков** — стандартный бенчмарк для задач кардиологической диагностики.

---
## Задание 2. Выбор метрики

### 2.1 Предлагаемая метрика

В качестве основной метрики предлагается **ROC-AUC**, в качестве дополнительной — **Recall (полнота)**.

**Обоснование:** классы в датасете практически сбалансированы (~54% с болезнью сердца, ~46% без), однако в задаче медицинской диагностики цена ошибки **FN** (пропустить больного пациента) несравнимо выше цены ошибки **FP** (направить здорового на доп. обследование). Пропущенная ишемическая болезнь сердца — риск инфаркта и смерти. Поэтому **Recall** по классу 1 (болезнь) является приоритетной метрикой: мы хотим поймать как можно больше больных. **ROC-AUC** позволяет оценить качество разделения классов в целом и настроить порог классификации под нужный баланс Precision/Recall.

---
## Задание 3. EDA (Разведочный анализ данных)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib import pyplot as plt

%matplotlib inline
sns.set_theme()

### 3.1 Загрузка данных

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/kb22/Heart-Disease-Prediction/master/dataset.csv'

df = pd.read_csv(DATA_URL)

print(f'Датасет загружен: {df.shape[0]} строк, {df.shape[1]} столбцов')
df.head()

### 3.2 Базовые характеристики датасета

In [ ]:
df.info()

**Вывод:** все 13 признаков числовые (`int64` или `float64`), явных пропусков нет. Датасет компактный (303 строки), но достаточен для бинарной классификации. Категориальные по смыслу признаки (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `thal`) закодированы как целые числа — при необходимости их можно преобразовать в one-hot для линейных моделей.

In [ ]:
df.describe()

**Вывод:** признаки находятся в очень разных масштабах — `chol` может достигать 564 мг/дл, а `oldpeak` лежит в диапазоне 0–6.2. Перед обучением линейных моделей обязательно применить `StandardScaler`. Максимальное давление `trestbps` = 200 и холестерин `chol` = 564 — потенциальные выбросы, требующие проверки.

### 3.3 Распределение целевой переменной

In [ ]:
target_counts = df['target'].value_counts().reset_index()
target_counts.columns = ['target', 'count']
target_counts['label'] = target_counts['target'].map({0: 'Нет болезни сердца', 1: 'Есть болезнь сердца'})

fig = px.pie(
    target_counts,
    values='count',
    names='label',
    title='Распределение целевой переменной target',
    color_discrete_map={'Есть болезнь сердца': '#F28B82', 'Нет болезни сердца': '#8BC4E0'},
    hole=0.35
)
fig.update_traces(textinfo='percent+label')
fig.show()

**Вывод:** классы практически сбалансированы — ~54% пациентов имеют болезнь сердца, ~46% здоровы. Это хорошая ситуация для обучения модели. Тем не менее ROC-AUC и Recall остаются предпочтительными метриками из-за медицинского контекста задачи.

### 3.4 Максимальная ЧСС — ключевой предиктор

In [ ]:
fig = px.histogram(
    df,
    x='thalach',
    color='target',
    nbins=30,
    barmode='overlay',
    opacity=0.7,
    title='Распределение максимальной ЧСС по наличию болезни сердца',
    labels={'thalach': 'Макс. ЧСС (уд./мин)', 'target': 'Болезнь сердца'},
    color_discrete_map={0: '#8BC4E0', 1: '#F28B82'}
)
fig.show()

**Вывод:** максимальная ЧСС (`thalach`) имеет **обратную** зависимость с наличием болезни — у больных пациентов ЧСС ниже (пик ~140–150 уд./мин), у здоровых — выше (пик ~160–170). Это объясняется тем, что при ишемической болезни сердца резервы сердечно-сосудистой системы снижены. `thalach` — один из наиболее информативных признаков.

### 3.5 Распределения всех числовых признаков

In [ ]:
num_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.histplot(
        data=df,
        x=col,
        hue='target',
        bins=25,
        palette={0: '#8BC4E0', 1: '#F28B82'},
        alpha=0.6,
        ax=axes[i]
    )
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].legend(title='Болезнь', labels=['Нет', 'Есть'], fontsize=8)

# Скрываем лишнюю ось
axes[-1].set_visible(False)

plt.suptitle('Распределения числовых признаков (синий = норма, красный = болезнь)', y=1.01)
plt.tight_layout()
plt.show()

**Вывод:**
- `thalach` — наиболее чёткое разделение: у больных пациентов ЧСС ниже.
- `oldpeak` — у больных значения выше (депрессия ST выражена сильнее), сильный правый скос.
- `age` — больные пациенты в среднем старше, но перекрытие классов значительное.
- `chol` и `trestbps` — распределения схожи между классами, признаки менее информативны.
- `oldpeak` имеет сильный правый скос — стоит применить `np.log1p()`-трансформацию.

### 3.6 Боксплоты: числовые признаки по классам

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.boxplot(
        data=df,
        x='target',
        y=col,
        palette={0: '#8BC4E0', 1: '#F28B82'},
        ax=axes[i]
    )
    axes[i].set_title(col)
    axes[i].set_xlabel('target (0=норма, 1=болезнь)')

axes[-1].set_visible(False)

plt.suptitle('Боксплоты числовых признаков по классу target')
plt.tight_layout()
plt.show()

**Вывод:**
- `thalach` — медиана у больных (~145) заметно ниже, чем у здоровых (~162). Высокая разделяющая способность.
- `oldpeak` — медиана у больных выше (~1.5 против ~0), присутствуют выбросы — рекомендуется log-трансформация.
- `age` — медианы различаются (~55 vs ~52), выбросов немного.
- `chol` — медианы практически совпадают (~246 vs ~235), признак слабо дискриминирует.
- `trestbps` — аналогично `chol`: медианы близки, высокий разброс в обоих классах.

### 3.7 Категориальные признаки: тип боли в груди и болезнь сердца

In [ ]:
cp_labels = {0: 'Типичная стенокардия', 1: 'Атипичная стенокардия', 2: 'Неангинальная боль', 3: 'Бессимптомная'}
df_cp = df.copy()
df_cp['cp_label'] = df_cp['cp'].map(cp_labels)

cp_rate = df_cp.groupby('cp_label')['target'].agg(['mean', 'count']).reset_index()
cp_rate.columns = ['cp_label', 'rate', 'count']

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(x=cp_rate['cp_label'], y=cp_rate['count'],
           name='Кол-во пациентов', marker_color='lightblue', opacity=0.6),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=cp_rate['cp_label'], y=cp_rate['rate'],
               name='Доля с болезнью сердца',
               mode='lines+markers',
               line=dict(color='#F28B82', width=2),
               marker=dict(size=10)),
    secondary_y=True
)

fig.update_layout(title='Тип боли в груди и распространённость болезни сердца')
fig.update_yaxes(title_text='Количество пациентов', secondary_y=False)
fig.update_yaxes(title_text='Доля с болезнью сердца', secondary_y=True, range=[0, 1])
fig.show()

**Вывод:** наблюдается неочевидная закономерность — у пациентов **без** типичной стенокардии (типы 1, 2, 3) болезнь сердца встречается значительно чаще (~70–80%), чем у пациентов с типичной стенокардией (тип 0, ~28%). Это парадокс: «бессимптомные» пациенты в этом датасете чаще больны. Признак `cp` — один из самых важных, его следует включить в модель как категориальный (one-hot encoding).

### 3.8 Корреляционная матрица

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    mask=mask,
    linewidths=0.5,
    square=True
)
plt.title('Корреляционная матрица признаков')
plt.tight_layout()
plt.show()

**Вывод:**
- Наибольшая корреляция с `target`: `cp` (+0.43), `exang` (+0.44), `oldpeak` (+0.43), `ca` (+0.38), `thal` (+0.34).
- `thalach` имеет **отрицательную** корреляцию с `target` (~−0.42): у пациентов с болезнью сердца максимальная ЧСС ниже — это физиологически логично (сниженный сердечный резерв).
- `exang`, `oldpeak`, `ca` и `thal` **положительно** коррелируют с `target`: более высокие значения этих признаков ассоциированы с наличием болезни сердца.
- `chol` и `trestbps` имеют очень слабую корреляцию с `target` (~0.09 и ~0.14) — признаки малоинформативны.
- Мультиколлинеарность умеренная, явных проблем нет.

### 3.9 Scatter plot: разделение классов по ключевым признакам

In [ ]:
df_plot = df.copy()
df_plot['Диагноз'] = df_plot['target'].map({0: 'Норма', 1: 'Болезнь сердца'})

fig = px.scatter(
    df_plot,
    x='thalach',
    y='oldpeak',
    color='Диагноз',
    color_discrete_map={'Болезнь сердца': '#F28B82', 'Норма': '#8BC4E0'},
    opacity=0.65,
    size_max=8,
    title='Разделение классов: макс. ЧСС vs депрессия ST (oldpeak)',
    labels={'thalach': 'Макс. ЧСС (уд./мин)', 'oldpeak': 'Депрессия ST (oldpeak)'}
)
fig.show()

**Вывод:** пара `thalach` + `oldpeak` хорошо разделяет классы. Пациенты с болезнью сердца (красные) концентрируются в левом верхнем углу — низкая ЧСС и высокая депрессия ST. Здоровые (синие) — в правом нижнем: высокая ЧСС, минимальная депрессия ST. Граница раздела нелинейная, что говорит в пользу ансамблевых методов (Random Forest, Gradient Boosting).

### 3.10 Попарные зависимости ключевых признаков

In [ ]:
key_features = ['age', 'thalach', 'oldpeak', 'chol', 'target']

sns.pairplot(
    df[key_features],
    hue='target',
    palette={0: '#8BC4E0', 1: '#F28B82'},
    plot_kws={'alpha': 0.5},
    diag_kind='kde'
)
plt.suptitle('Попарные зависимости ключевых признаков', y=1.02)
plt.show()

**Вывод:** наиболее чёткое разделение классов дают пары `thalach–oldpeak` и `age–thalach`. Признак `chol` практически не разделяет классы ни в одной паре — это подтверждает его слабую информативность. KDE-диагонали показывают, что у больных пациентов `thalach` имеет более узкое, смещённое влево распределение, а `oldpeak` — более широкое с правым хвостом.

---
## Итоги EDA и выводы для предобработки

**Пропуски:** отсутствуют — датасет полностью чистый.

**Выбросы:**
- `chol` (макс. 564 мг/дл), `trestbps` (макс. 200 мм рт. ст.) — проверить клинически; можно применить клиппинг на уровне 99-го перцентиля.
- `oldpeak` — сильный правый скос, рекомендуется `np.log1p()`-трансформация.

**Масштабирование:** обязательно применить `StandardScaler` — признаки в очень разных диапазонах.

**Категориальные признаки:** `cp`, `restecg`, `thal` — применить one-hot encoding для линейных моделей; для деревьев оставить как числа.

**Слабые признаки:** `chol` и `trestbps` — минимальная корреляция с `target`, при отборе признаков можно рассмотреть исключение.

**Ключевые признаки для модели:** `thalach`, `cp`, `exang`, `oldpeak`, `ca`, `thal`.